In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint05_Backtesting_Engine

Purpose:
Run all four strategies through the backtest engine: transaction
costs, slippage, and walk-forward validation. Compare gross vs net
of costs, and check performance across different stretches of
history rather than one full-history number.

Author:
Alison

Version:
0.7
"""

In [ ]:
# =====================================================
# ALPHA v0.7
# Sprint 5 - Backtesting Engine
# =====================================================

import matplotlib.pyplot as plt
import pandas as pd

from alpha.config import Config, DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.portfolio import calculate_monthly_returns
from alpha.regime import calculate_regime
from alpha.backtest import run_backtest, run_walk_forward_backtest

from alpha.strategies.momentum import calculate_monthly_momentum, select_top_stocks, select_bottom_stocks
from alpha.strategies.trend_following import select_trend_positions, select_downtrend_positions
from alpha.strategies.mean_reversion import select_oversold_stocks, select_overbought_stocks
from alpha.strategies.breakout import select_breakout_stocks, select_breakdown_stocks

In [ ]:
config = DEFAULT_CONFIG

# Same config but zero costs, so we can isolate the cost impact below
no_cost_config = Config(**{**config.__dict__, "transaction_cost_bps": 0, "slippage_bps": 0})

config

In [ ]:
# =====================================================
# DATA
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)
monthly_returns = calculate_monthly_returns(monthly_prices)

print("Downloading benchmark for regime filter...")

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

In [ ]:
# =====================================================
# RAW SIGNALS (unshifted - run_backtest shifts internally)
# =====================================================

monthly_momentum = calculate_monthly_momentum(monthly_prices, config)

signal_pairs = {
    "Momentum": (
        select_top_stocks(monthly_momentum, config),
        select_bottom_stocks(monthly_momentum, config),
    ),
    "Trend Following": (
        select_trend_positions(monthly_prices, config),
        select_downtrend_positions(monthly_prices, config),
    ),
    "Mean Reversion": (
        select_oversold_stocks(monthly_prices, config),
        select_overbought_stocks(monthly_prices, config),
    ),
    "Breakout": (
        select_breakout_stocks(monthly_prices, config),
        select_breakdown_stocks(monthly_prices, config),
    ),
}

In [ ]:
# =====================================================
# GROSS vs NET OF COSTS
# =====================================================
# This is the number that matters most from this sprint: how much of
# each strategy's return gets eaten by turnover.

rows = []
for name, (long_signal, short_signal) in signal_pairs.items():
    gross = run_backtest(monthly_returns, long_signal, short_signal, regime, no_cost_config)
    net = run_backtest(monthly_returns, long_signal, short_signal, regime, config)

    rows.append({
        "Strategy": name,
        "Gross Growth": gross.final_growth,
        "Net Growth": net.final_growth,
        "Cost Drag": net.total_cost_drag,
        "Avg Monthly Turnover": net.turnover.mean(),
        "Avg Holdings": net.avg_holdings,
    })

cost_summary = pd.DataFrame(rows).set_index("Strategy")
display(cost_summary)

In [ ]:
# =====================================================
# WALK-FORWARD VALIDATION
# =====================================================
# Is each strategy's performance consistent across different stretches
# of history, or driven by one lucky window?

for name, (long_signal, short_signal) in signal_pairs.items():
    print(f"\n{name}")
    wf = run_walk_forward_backtest(
        monthly_returns, long_signal, short_signal, regime, config,
        train_months=36, test_months=12, step_months=12,
    )
    display(wf)

In [ ]:
# =====================================================
# NET-OF-COST GROWTH, ALL STRATEGIES
# =====================================================

plt.figure(figsize=(12, 6))

for name, (long_signal, short_signal) in signal_pairs.items():
    result = run_backtest(monthly_returns, long_signal, short_signal, regime, config)
    result.growth.plot(label=name)

plt.title("Sprint 5 - Net-of-Cost Performance, All Strategies")
plt.xlabel("Date")
plt.ylabel("Portfolio Growth")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =====================================================
# NOTES
# =====================================================
# Cost assumptions (transaction_cost_bps + slippage_bps) live in
# Config - they're a rough retail estimate, not a promise. Worth
# tightening once you know your actual broker's fees and the
# liquidity of what you're trading.
#
# Walk-forward here doesn't fit any parameters on the train window -
# none of the current strategies have anything to fit. It's still
# useful as a sanity check: a strategy whose final_growth swings
# wildly between windows is telling you its edge (if any) isn't
# reliable, no matter how good the full-history number looks.
#
# Next is Sprint 6 - proper performance statistics (CAGR, Sharpe,
# Sortino, max drawdown, profit factor, expectancy) instead of eyeing
# a growth chart and a single final number.